In [14]:
!pip install torch torchvision torchaudio
!pip install torch-geometric


In [18]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
# ✅ 1. 데이터 준비

# KOPIS 폴더 경로 지정
base_path = "/content/drive/MyDrive/KOPIS"
data = torch.load(f"{base_path}/Btype_gnn_data.pt", weights_only=False)

print(data)
print("노드 수:", data.num_nodes)
print("엣지 수:", data.num_edges)
print("특성 차원:", data.num_node_features)

# y = 장르 노드에는 장르 레이블, 나머지 노드는 -1
mask = data.y != -1
labels = data.y[mask]

# 학습/검증 데이터 분리 (80:20 split)
node_indices = torch.arange(len(data.y))[mask]
perm = torch.randperm(node_indices.size(0))
train_size = int(0.8 * len(perm))

train_idx = node_indices[perm[:train_size]]
test_idx  = node_indices[perm[train_size:]]

Data(x=[87758, 4], edge_index=[2, 1405574], y=[87758])
노드 수: 87758
엣지 수: 1405574
특성 차원: 4


In [21]:
# ✅ 2. 모델 정의
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

# 모델 초기화
num_classes = len(torch.unique(labels))
model = GCN(in_channels=data.x.shape[1], hidden_channels=64, out_channels=num_classes)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [22]:
# ✅ 3. 학습 루프
for epoch in range(1, 101):  # 100 epoch
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)

    loss = F.cross_entropy(out[train_idx], data.y[train_idx])
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        model.eval()
        pred = out[test_idx].argmax(dim=1)
        acc = accuracy_score(data.y[test_idx].cpu(), pred.cpu())
        f1 = f1_score(data.y[test_idx].cpu(), pred.cpu(), average="weighted")
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")

Epoch 010 | Loss: 1.8459 | Acc: 0.0000 | F1: 0.0000
Epoch 020 | Loss: 1.7962 | Acc: 0.0000 | F1: 0.0000
Epoch 030 | Loss: 1.7925 | Acc: 0.0000 | F1: 0.0000
Epoch 040 | Loss: 1.7921 | Acc: 0.0000 | F1: 0.0000
Epoch 050 | Loss: 1.7920 | Acc: 0.0000 | F1: 0.0000
Epoch 060 | Loss: 1.7920 | Acc: 0.0000 | F1: 0.0000
Epoch 070 | Loss: 1.7920 | Acc: 0.0000 | F1: 0.0000
Epoch 080 | Loss: 1.7921 | Acc: 0.0000 | F1: 0.0000
Epoch 090 | Loss: 1.7921 | Acc: 0.0000 | F1: 0.0000
Epoch 100 | Loss: 1.7921 | Acc: 0.0000 | F1: 0.0000


In [23]:
# ✅ 4. 학습 완료 후 최종 평가
model.eval()
out = model(data.x, data.edge_index)
pred = out[test_idx].argmax(dim=1)

final_acc = accuracy_score(data.y[test_idx].cpu(), pred.cpu())
final_f1 = f1_score(data.y[test_idx].cpu(), pred.cpu(), average="weighted")

print("최종 정확도 (Accuracy):", final_acc)
print("최종 F1-score:", final_f1)

최종 정확도 (Accuracy): 0.0
최종 F1-score: 0.0


In [24]:
num_classes = len(torch.unique(data.y[data.y != -1]))
print("클래스 수:", num_classes)


클래스 수: 8


In [ ]:
# ✅ 5. 학습된 임베딩 저장 (추후 복합 장르 탐지용)
embeddings = model(data.x, data.edge_index).detach()
torch.save(embeddings, "Btype_node_embeddings.pt")
print("임베딩 저장 완료:", embeddings.shape)